In [1]:
import os
from typing import Iterable, List, Dict, Optional, Tuple, Union
import pandas as pd
import numpy as np
import xml.etree.ElementTree as et
import re

In [2]:
# -----------------------------
    # PSC - CyMx
# -----------------------------
def process_equipment(equipment, output_data):
    """
    Processes a single equipment entry and adds it to the output data.

    Args:
        equipment: Dictionary containing equipment information
        output_data: List to append the processed equipment data
    """
    if not equipment.get('equipment_type') or not equipment.get('sw_configs'):
        return

    for lri in equipment.get('lris', []):
        for sw_config in equipment['sw_configs']:
            # Find the latest hardware version (hw_num > 1)
            hw_versions = [hv for hv in sw_config.get('hw_versions', [])
                         if hv.get('hw_num', 0) > 1]
            hw_version = hw_versions[-1] if hw_versions else None

            # Create the output row
            row = {
                'equipment_type': equipment['equipment_type'],
                'lri_type_code': equipment.get('lri_type_code', ''),
                'lri_num': lri.get('lri_num', ''),
                'ident_code': lri.get('ident_code', ''),
                'sw_part_number': sw_config.get('sw_part_number', ''),
                'sw_modification_code': sw_config.get('sw_modification_code', ''),
                'sw_modification_status': 0,  # Default value
                'hw_part_number': hw_version.get('hw_part_number', '') if hw_version else '',
                'hw_part_number_code': hw_version.get('hw_part_number_code', '') if hw_version else ''
            }
            output_data.append(row)

In [ ]:
def parse_equipment_file(file_path):
    equipment_data = []
    current_equipment = None

    # Helper function to flatten the nested structure into rows
    def save_current_equipment(equipment):
        if not equipment or not equipment.get('lris'):
            return
        
        for lri in equipment['lris']:
            for sw_config in equipment['sw_configs']:
                # If no HW_VERSIONS exist, we still want the SW info
                # We use a dummy list if hw_versions is empty to ensure the loop runs once
                hvs = sw_config['hw_versions'] if sw_config['hw_versions'] else [None]
                
                for hv in hvs:
                    row = {
                        'equipment_type': equipment['equipment_type'],
                        'lri_type_code': equipment['lri_type_code'],
                        'lri_num': lri['lri_num'],
                        'ident_code': lri.get('ident_code'),
                        'sw_part_number': sw_config.get('sw_part_number'),
                        'sw_modification_code': sw_config.get('sw_modification_code'),
                        'sw_modification_status': 0,
                        'hw_part_number': hv.get('hw_part_number') if hv else None,
                        'hw_part_number_code': hv.get('hw_part_number_code') if hv else None
                    }
                    equipment_data.append(row)

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            # When we see "EQUIPMENT", save the PREVIOUS one and reset
            if line == "EQUIPMENT":
                save_current_equipment(current_equipment)
                current_equipment = {
                    'equipment_type': None,
                    'lri_type_code': None,
                    'lris': [],
                    'sw_configs': []
                }
                continue

            # Ensure current_equipment is initialized if file doesn't start with "EQUIPMENT"
            if current_equipment is None:
                current_equipment = {'equipment_type': None, 'lri_type_code': None, 'lris': [], 'sw_configs': []}

            # Parse properties
            if line.startswith(".equipment_type"):
                current_equipment['equipment_type'] = line.split("=>")[1].strip()
            elif line.startswith(".lri_type_code"):
                current_equipment['lri_type_code'] = line.split("=>")[1].strip()

            elif line.startswith("LRI_NO"):
                lri_num = int(re.search(r'LRI_NO\((\d+)\)', line).group(1))
                current_equipment['lris'].append({'lri_num': lri_num})

            elif line.startswith(".ident_code"):
                current_equipment['lris'][-1]['ident_code'] = line.split("=>")[1].strip()

            elif line.startswith("SW_CONFIG"):
                sw_num = int(re.search(r'SW_CONFIG\((\d+)\)', line).group(1))
                current_equipment['sw_configs'].append({
                    'sw_num': sw_num,
                    'sw_part_number': None,
                    'sw_modification_code': None,
                    'hw_versions': []
                })

            elif line.startswith(".sw_part_number"):
                current_equipment['sw_configs'][-1]['sw_part_number'] = line.split("=>")[1].strip()
            elif line.startswith(".sw_modification_code"):
                current_equipment['sw_configs'][-1]['sw_modification_code'] = line.split("=>")[1].strip()

            elif line.startswith("HW_VERSIONS"):
                hw_num = int(re.search(r'HW_VERSIONS\((\d+)\)', line).group(1))
                current_equipment['sw_configs'][-1]['hw_versions'].append({'hw_num': hw_num})
            
            elif line.startswith(".hw_part_number_code"):
                current_equipment['sw_configs'][-1]['hw_versions'][-1]['hw_part_number_code'] = line.split("=>")[1].strip()
            elif line.startswith(".hw_part_number"):
                current_equipment['sw_configs'][-1]['hw_versions'][-1]['hw_part_number'] = line.split("=>")[1].strip()

    # CRITICAL: Save the very last equipment after the loop ends
    save_current_equipment(current_equipment)

    return pd.DataFrame(equipment_data)

In [ ]:
# FUENTE UTILIZADA
df_cymx = parse_equipment_file(r'\\gfa60001\ILS\TAOSD1\Data Integration\EF\EE_ESS\CyMx\1_T2_T3\PSC_36_(P3Ec)\03_PSC36.76.00_definitive\PSC36_76_00.GLU') #deberiamso añadir vor y cius
df_cymx.to_excel(r'C:\Users\C06839\Desktop\psc_lri.xlsx', sheet_name='Sheet1', index=False)

In [5]:
file_path = input("👉 Introduce la ruta del archivo de Texto (.GLU) para CyMx: ").strip()

try:
    if file_path.endswith(('.GLU')):
        df_cymx = parse_equipment_file(file_path)
    else:
        raise ValueError("Formato no soportado. Usa .GLU")

    print(f"\n✅ Archivo cargado correctamente: {file_path}")

except FileNotFoundError:
    print("❌ No se encontró el archivo. Verifica la ruta.")
except Exception as e:
    print(f"⚠️ Ocurrió un error: {e}")

# VER EN QUÉ CARPETA DE RED SE QUIERE GUARDAR
df_cymx.to_excel(r'C:\Users\C06839\Desktop\psc_lri.xlsx', sheet_name='Sheet1', index=False)


✅ Archivo cargado correctamente: \\gfa60001\ILS\TAOSD1\Data Integration\EF\EE_ESS\CyMx\1_T2_T3\PSC_36_(P3Ec)\01_PSC36.76.00_GroundUseOnly\PSC_QUAD_ST1A_GROUND_USE.GLU


In [6]:
# -----------------------------
    # AC
# -----------------------------
#falta elegir OJOO
# #tail number
matricula='CE.16-15'#'0C.16-80'
#internal=industry tail number
msn ='ST0015'#'SS0061'
applicability = "ST 15"

In [7]:
# -----------------------------
    # BS
# -----------------------------
# elegir el BS (FALTA OJO)
#\\gfa60001\ILS\TAOSD1\Data Integration\EF\EE_ESS\BuildStandard\T2_T3\P3Ec\02_PSC36.76.00_VOR_CIUs\BS_PSC36.76.00_SS\2.Ref Data XML\2.2.Ref Data Log SW
workbook_bs_xml = r'\\gfa60001\ILS\TAOSD1\Data Integration\EF\EE_ESS\BuildStandard\T2_T3\P3Ec\01_PSC36.76.00\BS PSC36.76.00 ST\2.Ref Data XML\2.2.Ref Data Log SW\REF-Buildstd_LOG-REF-Buildstd_121125102519.xml'
tree = et.parse(workbook_bs_xml)
root = tree.getroot()


In [8]:
dataframes = {}

for table in root:

    table_name = table.attrib.get('TableName')

    # -----------------------------
    # Obtener nombres de columnas
    # -----------------------------
    columns = []
    columns_node = table.find('Columns')

    if columns_node is not None:
        for col in columns_node:
            col_name = col.attrib.get('name')
            columns.append(col_name)

    # -----------------------------
    # Leer SOLO los Row (datos)
    # -----------------------------
    rows = []

    for row in table.findall('Row'):   # 👈 CLAVE: solo Row
        row_data = {}

        for i, field in enumerate(row):
            value = field.text.strip() if field.text else None

            # usar nombres reales de columnas
            if i < len(columns):
                row_data[columns[i]] = value
            else:
                row_data[field.tag] = value

        rows.append(row_data)

    df = pd.DataFrame(rows)

    dataframes[table_name] = df


df_bs_type = dataframes.get('BUILD_STAND_TYPE')  
df_bs = dataframes.get('BUILD_STAND') 
df_bs_prtn = dataframes.get('BUILD_STAND_PRTN')  

In [9]:
#expor BS excel file
bs_df_sheets = {
    'BUILD_STAND_TYPE': df_bs_type,
    'BUILD_STAND': df_bs,
    'BUILD_STAND_PRTN': df_bs_prtn
}

with pd.ExcelWriter(r'C:\Users\C06839\Desktop\BS.xlsx') as writer:
    for sheet_name, df in bs_df_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [10]:
# -----------------------------
    # CEE
# -----------------------------
# elegir el CEE (FALTA OJO)
#CEE SS61 del 11/2/26
workbook_cee = r'C:\Users\C06839\Desktop\ESS_Axis\xmls_crearAC\CEE_ST15.xlsx'
df_cee = pd.read_excel(workbook_cee, dtype=str, keep_default_na=False,na_filter=False)

In [11]:
# -----------------------------
    # SPS, Motores y DECMUs
# -----------------------------
workbook_sps_motores_apus = r'C:\Users\C06839\Desktop\ESS_Axis\xmls_crearAC\template-SPS - Motores y DECMUs.xlsx'
sps_motores_apus = pd.read_excel(workbook_sps_motores_apus, sheet_name="SPS-ENGINES", dtype=str, keep_default_na=False,na_filter=False)

In [12]:
# -----------------------------
    # LRIs > 1 (particiones)
# -----------------------------
# #LRI MP OJO match con la bs 3 tabla!!
workbook_lri_mp = r'\\gfa60001\ILS\TAOSD1\Data Integration\EF\EE_ESS\LRI Mapping Table\2025_Halcon_I_PSC_36.76.00\PSC36_76_00_AdaptedbyADS_SP.xlsx'
df_lri_mp = pd.read_excel(workbook_lri_mp, sheet_name="Mapping table", dtype=str, skiprows=11,
    header=0,keep_default_na=False,na_filter=False)
df_lri_mp['Partition No'] = pd.to_numeric(df_lri_mp['Partition No'], errors='coerce')
#tener variable para esto! OJO FALTA
df_lri_mp['LRI_IDENT'] = df_lri_mp['LRI Ident\n(dec)']
df_lri_mp_filtered = df_lri_mp[(df_lri_mp['Variant'] != 'TWIN')]
df_lri_completa = df_lri_mp[(df_lri_mp['Variant'] != 'TWIN')].copy()
df_lri_completa['Nº Partitions'] = df_lri_completa.groupby("LCN")["LCN"].transform("count")
df_lri_completa.to_excel(r'C:\Users\C06839\Desktop\LRI_MP_completa.xlsx', sheet_name='Sheet1', index=False)
df_lri_mp_filtered_lri = df_lri_mp[(df_lri_mp['Partition No'] > 1) & (df_lri_mp['Variant'] != 'TWIN')]
df_lri_mp_filtered_lri.to_excel(r'C:\Users\C06839\Desktop\LRI_MP.xlsx', sheet_name='Sheet1', index=False)

In [13]:
# --------------------------------------------
    # ASSET - formar lcn-hw pnr - sn - sw pnr
# --------------------------------------------

result_bs = df_bs 
result_bs = pd.merge(df_bs, df_lri_mp_filtered, on='LRI_IDENT', how='left')
result_bs.rename(columns={'LCN_x':'LCN', 'LCN_y': 'LCN_LRI_MP'},inplace=True)
result_bs['HW_PART_NO_BS'] = result_bs['HW_PART_NO']
result_bs = result_bs.rename(columns={'HW_PART_NO': 'HW_PNR'}) 
result_bs = result_bs.drop(result_bs[result_bs['LRI_IDENT'] == '0'].index)

df_cee = df_cee.rename(columns={'PARTNUMBER': 'HW_PNR'})
merged_df = pd.merge(result_bs, df_cee, on=['LCN', 'HW_PNR'], how='left')
asset_aux = merged_df.loc[:, ['LCN','ALCN','HW_PART_NO_BS','HW_PNR','SERIALNBR','ITEM_NAME','LRI_IDENT','LRI Ident\n(hex)']]

asset_sn_null = asset_aux[asset_aux['SERIALNBR'].isna()]
asset_sn_null = asset_sn_null.groupby('LCN', as_index=False).last()

asset_aux = asset_aux.dropna(subset=['SERIALNBR'])

#quitar de asset_sn_null los LCN que ya esten en asset_aux
asset_aux_LCNs = asset_aux['LCN'].unique()
asset_sn_null=asset_sn_null[~asset_sn_null['LCN'].isin(asset_aux_LCNs)]

#por si hay mas de una ALCN con SN para el mismo LCN-PNR que se quede con la ultima
asset_aux= asset_aux.groupby('LCN', as_index=False).last()

asset = pd.concat([asset_aux, asset_sn_null], ignore_index=True)


In [14]:
#para asociar cada sw al hw correspondiente se necesita el LRI Ident(hex) que viene
#en el nodo LRI de la PSC ya que hay equipos que son iguales pero tiene distinta funcionalidad 
#por tanto distinto SW y la unica forma de diferenciarlo es por los LRIs asociados
result_cymx = df_cymx

result_cymx = result_cymx.rename(columns={'hw_part_number': 'HW_PNR','ident_code':'LRI Ident\n(hex)'}) 
merged_asset = pd.merge(asset, result_cymx, on=['HW_PNR','LRI Ident\n(hex)'], how='inner')


In [15]:
merged_asset['ASSET_STATUS'] = 'FI'
merged_asset['STORES_LOCATION'] = msn
merged_asset.rename(columns={'SERIALNBR': 'SERIAL_NO','HW_PNR': 'HW_PART_NO','sw_part_number': 'LOADED_SOFTWARE','sw_modification_code': 'SW_PART_NO_CODE'},inplace=True)

asset = merged_asset[['SERIAL_NO', 'HW_PART_NO', 'LCN','STORES_LOCATION', 'ASSET_STATUS', 'LOADED_SOFTWARE','SW_PART_NO_CODE']]

In [16]:
def is_in_range(cell_value, target):
    if pd.isna(cell_value) or not isinstance(cell_value, str):
        return False
    
    # 1. Prepare the target (e.g., "ST 15" -> prefix="ST", num=15)
    try:
        target_parts = target.split()
        if len(target_parts) < 2:
            return False
        target_prefix = target_parts[0]
        target_num = int(target_parts[1])
    except ValueError:
        return False

    # 2. Split the cell into individual segments using ";"
    # Example: "SS 1-9; ST 1-99;" -> ["SS 1-9", " ST 1-99", ""]
    segments = cell_value.split(';')

    # 3. Iterate through each segment to see if ANY match the target
    for segment in segments:
        segment = segment.strip() # Remove leading/trailing whitespace
        if not segment:
            continue # Skip empty segments caused by trailing ";"
            
        # Split segment into prefix and range (e.g., "ST 1-99" -> ["ST", "1-99"])
        parts = segment.split(None, 1)
        if len(parts) < 2:
            continue
            
        cell_prefix = parts[0]
        cell_range = parts[1]

        # Check if this specific segment's prefix matches our target prefix
        if cell_prefix == target_prefix:
            try:
                # Parse the range (e.g., "1-99" -> 1 and 99)
                start_str, end_str = cell_range.split('-')
                start_val = int(start_str)
                end_val = int(end_str)
                
                # If the number falls in this range, we found a match!
                if start_val <= target_num <= end_val:
                    return True
            except ValueError:
                continue # If this segment is malformed, move to the next one
                
    # If we checked all segments and found no match
    return False

In [17]:
# ---------------------------------------------
    # MLCN: find LCN-PNR of SPSs to set TCIs 
# ---------------------------------------------
workbook_mlcn = r'C:\Users\C06839\Downloads\MLCN.xlsx'
df_mlcn = pd.read_excel(workbook_mlcn, sheet_name="Table", dtype=str, keep_default_na=False,na_filter=False)

sps_mlcn_df = df_mlcn[
    (df_mlcn['IERS'] == 'S') & 
    (df_mlcn['GSS_ITEM'] == 'S') & 
    (df_mlcn['LCN'].str.strip().str.upper().str.startswith(('X8', 'X49'), na=False))
]

filtered_df_mlcn_sps = sps_mlcn_df[sps_mlcn_df['EFEC_PN'].apply(lambda x: is_in_range(x, applicability))]

print(filtered_df_mlcn_sps)

          LCN ALC                     TITULO_LCN PART_NUMBER_MS  \
1647    X8324   A                AMAD GEARBOX,RH    DBEF8036-01   
2065    X8321   -                AMAD GEARBOX,LH    DBEF8037-01   
2261    X8321   A                AMAD GEARBOX,LH    DBEF8035-01   
2762  X491201   A               APU CONTROL UNIT    EFP18807-10   
3219    X8324   -                AMAD GEARBOX,RH    DBEF8038-01   
3415  X491201   -               APU CONTROL UNIT     EFP18807-9   
3511  X801102   -  AIR TURBINE STARTER MOTOR, LH     MDR1301-07   
3964  X801104   -   AIR TURBINE STARTER MOTOR,RH     MDR1301-07   
4069    X4911   B                   APU ASSEMBLY     EFP10004-6   

     COD_OTAN_FABRICANTE SDR EDR IDR GSS_ITEM  \
1647               K0648   A   A   D        S   
2065               A3460   A   A   D        S   
2261               A3460   A   A   D        S   
2762               C0548   A   A   D        S   
3219               K0648   A   A   D        S   
3415               C0548   A   A   

In [18]:
# -----------------------------------------------------
    # Equivalencias TCIs CMS vs ESS(Axis) vs SL2000 
# -----------------------------------------------------
workbook_tci_eq = r'C:\Users\C06839\Downloads\equivalencias TCI.xlsx'
df_tci_eq = pd.read_excel(workbook_tci_eq, sheet_name="Table", dtype=str, keep_default_na=False,na_filter=False)

workbook_tci_va_cms = r'C:\Users\C06839\Downloads\VidaAutorizada_20260828.xlsx'
df_tci_va_cms = pd.read_excel(workbook_tci_va_cms, sheet_name="Table", dtype=str, keep_default_na=False,na_filter=False)

#incluir en va su equivalencia ESS
df_tci_va_cms = df_tci_va_cms.merge(df_tci_eq[['TCI_PIN', 'TCI_ESS']], left_on='TCI', right_on='TCI_PIN', how='left')
df_tci_va_cms = df_tci_va_cms.drop(columns=['TCI_PIN'])
#si TCI = TCI_ESS, TCI_ESS era vacio--> rellenar con valor TCI
df_tci_va_cms['TCI_ESS'] = np.where(df_tci_va_cms['TCI_ESS'].isnull(), df_tci_va_cms['TCI'], df_tci_va_cms['TCI_ESS'])

print(df_tci_va_cms)
df_tci_va_cms.to_excel(r'C:\Users\C06839\Desktop\VA_eqESS.xlsx', sheet_name='Sheet1', index=False)

#cruzar con mlcn_sps y quedarnos con pn, tci y vida max. 
df_tci_cms = df_tci_va_cms.merge(
        filtered_df_mlcn_sps, 
        left_on='Parte', 
        right_on='PART_NUMBER_MS', 
        how='inner'
    )
print(df_tci_cms)

#eliminar FH = HV
df_tci_cms = df_tci_cms[df_tci_cms['TCI'] != 'HV']
df_tci_cms = df_tci_cms[['Parte', 'TCI_ESS', 'Vida Máxima']]

# Cambiar nombres a HW_PART_NO, LIFE_TYPE, EXPIRY_VALUE
df_tci = df_tci_cms.rename(columns={
    'Parte': 'HW_PART_NO',
    'TCI_ESS': 'LIFE_TYPE',
    'Vida Máxima': 'EXPIRY_VALUE'
})
df_tci.to_excel(r'C:\Users\C06839\Desktop\TCIs_cms.xlsx', sheet_name='Sheet1', index=False)

             Parte         Descripción Part Number Cod. OTAN Fabricante TCI  \
0          ABL10-5                    STROBE LIGHT                A2829  HV   
1         ACM30743               SHAFT, SHOULDERED                K1037  HV   
2         ACM31784          WHEEL SPEED TRANSDUCER                K1037  HV   
3         ACM31784          WHEEL SPEED TRANSDUCER                K1037  FH   
4         ACO41182                    SLIPPER SEAL                       FH   
...            ...                             ...                  ...  ..   
5536  9171A0015-01  HARNESS, MANIFOLD AND FEEDBACK                D9893  HV   
5537  9171A0015-03       CABLE ASSEMBLY-SWITCH, EL                D9893  HV   
5538         92425         FUEL COOL.OIL COOLER LH                D8093  HV   
5539         92426         FUEL COOL.OIL COOLER RH                D8093  HV   
5540  96-1-2100-05               PEDAL SENSOR UNIT                D2638  HV   

     Vida Autorizada Ind. TCI Principal Porcentaje 

In [19]:
# ---------------------------------------------
    # TCIs de los SPSs --> Vida Autorizada CMS
# ---------------------------------------------

df_tci = df_tci.rename(columns={'HW_PART_NO': 'HW_PNR'}) 

#solo se colocan los life type de los SPS que esten instalados
df_tci = pd.merge(df_tci, df_cee, on='HW_PNR', how='inner')
df_tci = df_tci.rename(columns={'HW_PNR': 'HW_PART_NO','SERIALNBR':'SERIAL_NO'}) 

In [20]:
# ----------------------------------
    # ASSET_LIFE_TYPE (TCIs de SPSs) --> VA!
# ----------------------------------
#como no se todos los valores de EXPIRY_VALUE, pongo a los que no tenga el maximo valor que dispongo=648000 (la excel usada), todos los valores de 
#CUMMULATIVE_TOTAL y METRIC_NUMBER a 0
df_tci['CUMMULATIVE_TOTAL'] = 0
df_tci['METRIC_NUMBER'] = 0
df_tci = df_tci[['LIFE_TYPE','LCN','HW_PART_NO', 'SERIAL_NO','CUMMULATIVE_TOTAL', 'METRIC_NUMBER', 'EXPIRY_VALUE']]
df_tci.to_excel(r'C:\Users\C06839\Desktop\asset_life_type.xlsx', sheet_name='Sheet1', index=False)

#Incluir las SPSs en el asset con sw vacio porque no son equipos electronicos (no emiten señales)
asset.to_excel(r'C:\Users\C06839\Desktop\asset_SS.xlsx', sheet_name='Sheet1', index=False)
tci_aux = df_tci[['LCN', 'HW_PART_NO', 'SERIAL_NO']].drop_duplicates()
tci_aux['ASSET_STATUS'] = 'FI'
tci_aux['STORES_LOCATION'] = msn

asset = pd.concat([asset, tci_aux], ignore_index=True)
asset.to_excel(r'C:\Users\C06839\Desktop\asset_SS_SPS.xlsx', sheet_name='Sheet1', index=False)


In [21]:
# -----------------------------------------
    # ASSET_PRTN --> LRIs > 1 (particiones)
# -----------------------------------------
#inner para lo que no este instalado no se ponga OJOO y match con df_bs_prtn
partition_aux = df_lri_completa[(df_lri_completa['Partition No'] == 1)& (df_lri_completa['Nº Partitions'] > 1)]
partition_aux=partition_aux[['LCN','Nº Partitions']]

result = (
    partition_aux[partition_aux['Nº Partitions'] > 1]  # Filter for LCNs with count > 1
    .assign(partition=lambda x: x['Nº Partitions'].apply(lambda c: list(range(2, c+1))))
    .explode('partition')
    .reset_index(drop=True)
    .rename(columns={'partitions': 'partition'})
)

partitions = result.merge(
    asset[['LCN', 'HW_PART_NO','SERIAL_NO','LOADED_SOFTWARE','SW_PART_NO_CODE']],  
    on='LCN',                    
    how='inner'                   
)

partitions = partitions.rename(columns={'partition': 'PARTITION_NO'}) 

In [22]:
# ---------------------------------------------
    # RELLENAR SN VACIOS EN ASSET Y ASSET_PRTN
# ---------------------------------------------
#rellenar SN vacios con 
partitions['SERIAL_NO'] = partitions['SERIAL_NO'].fillna('$' + partitions['LCN'].astype(str) + '_' + msn)
asset['SERIAL_NO'] = asset['SERIAL_NO'].fillna('$' + asset['LCN'].astype(str) + '_' + msn)
partitions.to_excel(r'C:\Users\C06839\Desktop\partitionWithSN.xlsx', sheet_name='Sheet1', index=False)
asset.to_excel(r'C:\Users\C06839\Desktop\asset_withSN.xlsx', sheet_name='Sheet1', index=False)

#Para axis sustituir todos los SN con 
partitions['SERIAL_NO'] = partitions.apply(lambda row: '$' + str(row['LCN']) + '_' + msn, axis=1)
asset['SERIAL_NO'] = asset.apply(lambda row: '$' + str(row['LCN']) + '_' + msn, axis=1)
partitions.to_excel(r'C:\Users\C06839\Desktop\partition_final.xlsx', sheet_name='Sheet1', index=False)
asset.to_excel(r'C:\Users\C06839\Desktop\asset_final.xlsx', sheet_name='Sheet1', index=False)

partitions = partitions.drop('Nº Partitions', axis=1)

In [23]:
# -----------------------------------------
    # Transformar a XML
# -----------------------------------------
#ASSET
asset
#ASSET_LIFE_TYPE
df_tci
#ASSET_PRTN
partitions

import xml.etree.ElementTree as ET

def create_field(fields,name, field_type, defined_size, precision, numeric_scale):
    field = ET.SubElement(fields,"Field")
    ET.SubElement(field, "Name").text = name
    ET.SubElement(field, "Type").text = str(field_type)
    ET.SubElement(field, "DefinedSize").text = str(defined_size)
    ET.SubElement(field, "Precision").text = str(precision)
    ET.SubElement(field, "NumericScale").text = str(numeric_scale)
    return field

def create_row(table, row_data):
    """Create a data row element"""
    row = ET.SubElement(table, "Row")
    for field_name, value in row_data.items():
        ET.SubElement(row, field_name).text = str(value)
    return row

# Create the root element
root = ET.Element("XML_Transfer")

# 1. ASSET table
asset_table = ET.SubElement(root, "Table")
asset_table.set("name", "ASSET")
asset_fields = [
    ("SERIAL_NO", 200, 30, 255, 255),
    ("HW_PART_NO", 200, 32, 255, 255),
    ("LCN", 200, 11, 255, 255),
    ("STORES_LOCATION", 129, 8, 255, 255),
    ("ASSET_STATUS", 129, 2, 255, 255),
    ("LOADED_SOFTWARE", 200, 32, 255, 255),
    ("SW_PART_NO_CODE", 129, 4, 255, 255),
]

fields = ET.SubElement(asset_table, "Fields")
for field_data in asset_fields:
    create_field(fields,*field_data)

# Add data rows
for _, row in asset.iterrows():
    row_dict = row.to_dict()
    create_row(asset_table, row_dict)


# 2. ASSET_LIFE_TYPE table
life_type_table = ET.SubElement(root, "Table")
life_type_table.set("name", "ASSET_LIFE_TYPE")
life_type_fields = [
    ("LIFE_TYPE", 129, 2, 255, 255),
    ("HW_PART_NO", 200, 32, 255, 255),
    ("SERIAL_NO", 200, 13, 255, 255),
    ("CUMMULATIVE_TOTAL", 131, 19, 14, 6),
    ("METRIC_NUMBER", 131, 19, 3, 0),
    ("EXPIRY_VALUE", 131, 19, 11, 3),
]

fields = ET.SubElement(life_type_table, "Fields")
for field_data in life_type_fields:
    create_field(fields,*field_data)

# Add data rows
for _, row in df_tci.iterrows():
    row_dict = row.to_dict()
    create_row(life_type_table, row_dict)

# 3. ASSET_PRTN table
prtn_table = ET.SubElement(root, "Table")
prtn_table.set("name", "ASSET_PRTN")
prtn_fields = [
    ("SERIAL_NO", 200, 30, 0, 0),
    ("HW_PART_NO", 200, 32, 0, 0),
    ("PARTITION_NO", 131, 19, 2, 0),
    ("LOADED_SOFTWARE", 200, 32, 0, 0),
    ("SW_PART_NO_CODE", 129, 4, 0, 0),
]

fields = ET.SubElement(prtn_table, "Fields")
for field_data in prtn_fields:
    create_field(fields,*field_data)

# Add data rows
for _, row in partitions.iterrows():
    row_dict = row.to_dict()
    create_row(prtn_table, row_dict)

# Create a pretty XML string
def pretty_xml(element, indent='  '):
    from xml.dom import minidom
    rough_string = ET.tostring(element, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent=indent)

# Save to file
xml_str = pretty_xml(root)
with open(r'C:\Users\C06839\Desktop\database_schema.xml', "w", encoding="utf-8") as f:
    f.write(xml_str)

print("XML file created successfully: database_schema.xml")

XML file created successfully: database_schema.xml


In [24]:
# -----------------------------------------
    # Report
# -----------------------------------------

#TCIs cms vs miExcel de SPS-GB para que salgan q nos faltan 4 pnr q vendran en el step 2 de halcon
workbook_manual_tci = r'C:\Users\C06839\Desktop\ESS_Axis\xmls_crearAC\template-SPS - Motores y DECMUs.xlsx'
df_manual_tci = pd.read_excel(workbook_manual_tci, sheet_name="TCI asset", dtype=str, keep_default_na=False,na_filter=False)

#TCIs cms vs vus por si falta alguno no nuevo que no tengamos en cms trazado y el ea si



#hacer report final, ordenar el codigo, solicitar ficheros en vez de ponerlo a mano, selecciona avion de un combo que las opcione ssean CE.16-15 - ST015... idem para aplicabilidad (lista de bloques)
#comparar contra lo de axis y contra cms en xmlspy y luego si se puede en codigo para el reporte tb

In [ ]:
# INAUGURAMOS INFORME DE SALIDA
outfile_name = create_outfile_name_excel(
    report_str="Analisis_huecos_SL2000_jupyter_" + avion + "_", weeknum=True
)

rdcd_wk_table = rdcd_wk_table_un_ac_id
huecos_table = df_sin_dupes_con_ac_conMDMD_h
inventario_table = df_sin_dupes_con_ac_conMDMD_inv
ac_inv_ent_table = inventario_ac_ent_id

deleted_desc_RDCD_table = df_eliminados

recept_M_table = disc_recep_M_final
recept_D_table = disc_recep_D_final
huecos_no_RDCD_table = check_h_no_rdcd_final
inv_no_RDCD_table = check_inv_no_rdcd_final
RDCD_M_table = check_rdcd_m_final
RDCD_D_table = check_rdcd_d_final
ok_M_vs_RDCD_Dates_table = ok_M_KO_dates
inv_vs_huecos_table = inv_vs_h

ok_M_table = ok_M_final
ok_D_table = ok_D_final

huecos_table_rows = huecos_table.shape[0]
rdcd_wk_table_rows = rdcd_wk_table.shape[0]
inventario_table_rows = inventario_table.shape[0]
ac_inv_ent_table_rows = ac_inv_ent_table.shape[0]

deleted_desc_RDCD_table_rows = deleted_desc_RDCD_table.shape[0]

recept_M_table_rows = recept_M_table.shape[0]
recept_D_table_rows = recept_D_table.shape[0]
huecos_no_RDCD_table_rows = huecos_no_RDCD_table.shape[0]
inv_no_RDCD_table_rows = inv_no_RDCD_table.shape[0]
RDCD_M_table_rows = RDCD_M_table.shape[0]
RDCD_D_table_rows = RDCD_D_table.shape[0]
ok_M_vs_RDCD_Dates_table_rows = ok_M_vs_RDCD_Dates_table.shape[0]
inv_vs_huecos_table_rows = inv_vs_huecos_table.shape[0]

ok_M_table_rows = ok_M_table.shape[0]
ok_D_table_rows = ok_D_table.shape[0]

table_debut(
            outfile_name=outfile_name,
            HUECOS_fpath=ruta_src_huecos,
            RDCD_fpath=ruta_src_rdcd,
            INVENTARIO_fpath=ruta_src_inv,
            AC_INV_E_fpath=ruta_src_ac_inv,
            output_folder=output_path,
            template_path=template_path,
            huecos_table_rows=huecos_table_rows,
            rdcd_wk_table_rows=rdcd_wk_table_rows,
            inventario_table_rows=inventario_table_rows,
            ac_inv_ent_table_rows=ac_inv_ent_table_rows,
            deleted_desc_RDCD_table_rows= deleted_desc_RDCD_table_rows,
            recept_M_table_rows = recept_M_table_rows,
            recept_D_table_rows = recept_D_table_rows,
            huecos_no_RDCD_table_rows = huecos_no_RDCD_table_rows,
            inv_no_RDCD_table_rows = inv_no_RDCD_table_rows,
            RDCD_M_table_rows = RDCD_M_table_rows,
            RDCD_D_table_rows = RDCD_D_table_rows,
            ok_M_vs_RDCD_Dates_table_rows = ok_M_vs_RDCD_Dates_table_rows,
            inv_vs_huecos_table_rows = inv_vs_huecos_table_rows, 
            ok_M_table_rows = ok_M_table_rows,
            ok_D_table_rows = ok_D_table_rows
        )

append_new_table(huecos_table, outfile_name, "src_MXCY", output_path)
append_new_table(inventario_table, outfile_name, "src_BS", output_path)
append_new_table(ac_inv_ent_table, outfile_name, "src_CEE", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_LRI_MP", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_SPS_ENGINES", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_VUS", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_VA", output_path) #estos con el de sps compararlo!
append_new_table(rdcd_wk_table, outfile_name, "src_CMS_BS", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_CMS_Asset", output_path)
append_new_table(rdcd_wk_table, outfile_name, "src_CMS_CyMx", output_path)

append_new_table(deleted_desc_RDCD_table, outfile_name, "deleted_rdcd_sem_observ", output_path)

append_new_table(recept_M_table, outfile_name, "recep_M", output_path)
append_new_table(recept_D_table, outfile_name, "recep_D", output_path)
append_new_table(huecos_no_RDCD_table, outfile_name, "huecos_no_RDCD", output_path)
append_new_table(inv_no_RDCD_table, outfile_name, "inv_no_RDCD", output_path)
append_new_table(RDCD_M_table, outfile_name, "RDCD_M", output_path)
append_new_table(RDCD_D_table, outfile_name, "RDCD_D", output_path)
append_new_table(ok_M_vs_RDCD_Dates_table, outfile_name, "ok_M_vs_RDCD_Dates", output_path)
append_new_table(inv_vs_huecos_table, outfile_name, "inv_vs_huecos", output_path)

append_new_table(ok_table, outfile_name, "ok", output_path)


